In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Load the epoch metrics and sweep summary data
df_epoch = pd.read_csv('epoch_metrics.csv')
df_sweep = pd.read_csv('sweep_summary.csv')

# Get unique models
models = df_epoch['model'].unique()

# Map long hyperparameter names to shorter abbreviations for the title
hp_mapping = {
    'lr': 'lr', 
    'dropout': 'drop', 
    'batch_size': 'bs', 
    'num_heads': 'heads', 
    'head_dim': 'h_dim', 
    'num_filters': 'filters', 
    'cnn_kernel_size': 'cnn_ks', 
    'lstur_mode': 'mode'
}

sns.set_theme(style="whitegrid")

for model_name in models:
    # Filter data for the current model
    model_df = df_epoch[df_epoch['model'] == model_name]
    
    # Unique run IDs for this model
    run_ids = sorted(model_df['run_id'].unique())
    num_runs = len(run_ids)
    
    if num_runs == 0:
        continue
        
    cols = 4
    rows = math.ceil(num_runs / cols)
    
    # If there are fewer than 4 runs, adjust columns to fit exactly
    if num_runs < 4:
        cols = num_runs
        
    # Create figure
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), squeeze=False)
    axes = axes.flatten()
    
    for i, run_id in enumerate(run_ids):
        ax = axes[i]
        run_data = model_df[model_df['run_id'] == run_id]
        first_row = run_data.iloc[0]
        
        # Plot Train and Val loss
        ax.plot(run_data['epoch'], run_data['train_loss'], label='Train Loss', marker='.')
        ax.plot(run_data['epoch'], run_data['val_loss'], label='Val Loss', marker='.')
        
        # Extract non-null hyperparameters
        hps = []
        for col, short_name in hp_mapping.items():
            val = first_row[col]
            if pd.notna(val):
                if isinstance(val, float) and val.is_integer():
                    val = int(val)
                hps.append(f"{short_name}={val}")
                
        # Join hyperparameters into a string. Split into two lines if it's too long.
        hp_str = ", ".join(hps)
        if len(hp_str) > 40: 
            split_idx = hp_str.find(", ", len(hp_str)//2)
            if split_idx != -1:
                hp_str = hp_str[:split_idx] + ",\n" + hp_str[split_idx+2:]
        
        # Title format: MODEL (Run X) \n hyperparams
        title = f"{str(model_name).upper()} (Run {run_id})\n{hp_str}"
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('Epoch', fontsize=9)
        ax.set_ylabel('Loss', fontsize=9)
        ax.tick_params(axis='both', which='major', labelsize=8)
        
        # Fetch metrics from sweep_summary for this run
        sweep_data = df_sweep[df_sweep['run_id'] == run_id]
        if not sweep_data.empty:
            s_row = sweep_data.iloc[0]
            # Create a text string with the key test metrics
            metrics_text = (
                f"Test AUC: {s_row['test_auc']:.4f}\n"
                f"Test MRR: {s_row['test_mrr']:.4f}\n"
                f"nDCG@5: {s_row['test_ndcg5']:.4f}\n"
                f"nDCG@10: {s_row['test_ndcg10']:.4f}"
            )
            
            # Place the metrics box in the upper right corner
            props = dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='gray')
            ax.text(0.95, 0.95, metrics_text, transform=ax.transAxes, fontsize=8,
                    verticalalignment='top', horizontalalignment='right', bbox=props)
            
            # Put the legend in the lower left to avoid the text box
            ax.legend(fontsize=8, loc='lower left')
        else:
            ax.legend(fontsize=8)

    # Hide any unused subplots
    for j in range(num_runs, len(axes)):
        fig.delaxes(axes[j])

    # Super title for the whole figure
    fig.suptitle(f"Training Curves & Final Metrics: {str(model_name).upper()}", fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    filename = f'{model_name}_curves_with_metrics.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)